# Karpathy — Building micrograd: Step-by-Step Walkthrough

**Video:** [The spelled-out intro to neural networks and backpropagation](https://www.youtube.com/watch?v=VMj-3S1tku0)

Covers all chapters:
- 08:08 — Derivative of one input
- 14:12 — Multiple inputs (partial derivatives)
- 19:09 — The `Value` object
- 32:10 — Manual backprop #1: simple expression
- 51:10 — What gradients mean / one optimization step
- 52:52 — Manual backprop #2: a neuron
- 01:09:02 — `_backward` for each op + shared nodes
- 01:17:32 — Automatic `backward()` via topological sort
- 01:45:00 — Full training loop
- 01:55:00 — Verify against PyTorch

In [ ]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1 — Derivative of a simple function (08:08)

The derivative f'(x) = lim_{h->0} [f(x+h) - f(x)] / h

Intuition: if I nudge x by a tiny h, how much does f change?

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

xs = np.arange(-5, 5, 0.25)
plt.plot(xs, f(xs))
plt.title('f(x) = 3x^2 - 4x + 5'); plt.grid(True); plt.show()

# Numerical derivative at x=3 (analytic: f'(3) = 6*3-4 = 14)
h = 1e-5
x = 3.0
print(f"f'(3) numerical = {(f(x+h)-f(x))/h:.5f}")
print(f"f'(3) analytic  = {6*x-4:.5f}")

# Derivative is 0 at the minimum: x = 2/3
x = 2/3
print(f"f'(2/3) = {(f(x+h)-f(x))/h:.8f}  (should be ~0)")

## 2 — Partial derivatives: multiple inputs (14:12)

Expression: d = a*b + c

dd/da = b, dd/db = a, dd/dc = 1

In [ ]:
a, b, c = 2.0, -3.0, 10.0
h = 1e-4
d = lambda a,b,c: a*b + c
d0 = d(a, b, c)
print(f'd = {d0}')  # 4.0
print(f'dd/da = {(d(a+h,b,c)-d0)/h:.4f}  (expected b = {b})')
print(f'dd/db = {(d(a,b+h,c)-d0)/h:.4f}  (expected a = {a})')
print(f'dd/dc = {(d(a,b,c+h)-d0)/h:.4f}  (expected 1.0)')

## 3 — The Value class (19:09)

We wrap every scalar in a Value that:
- stores .data (the number)
- stores .grad (dL/d_this, starts at 0)
- remembers ._prev (children in the computation graph)
- stores ._backward (function that propagates gradient to children)

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f'Value({self.label!r}, data={self.data}, grad={self.grad})'

    # ---- ops: forward + _backward closure ----

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            # local gradient of + is 1 for both inputs
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            # d(a*b)/da = b, d(a*b)/db = a
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,), f'**{other}')
        def _backward():
            # d(x^n)/dx = n * x^(n-1)
            self.grad += other * (self.data**(other-1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            # d(tanh)/dx = 1 - tanh^2
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')
        def _backward():
            # d(e^x)/dx = e^x
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    # derived ops built from primitives
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __neg__(self):         return self * -1
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * other**-1

    # ---- backward: topological sort + chain rule ----

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)   # leaves first, root last
        build(self)
        self.grad = 1.0          # dL/dL = 1
        for node in reversed(topo):  # root first → leaves last
            node._backward()

print('Value class defined OK')

## 4 — Manual backprop #1: L = (a*b + c) * f  (32:10)

Chain rule example:
```
e = a*b  →  de/da=b,   de/db=a
d = e+c  →  dd/de=1,   dd/dc=1
L = d*f  →  dL/dd=f,   dL/df=d

dL/da = dL/dd * dd/de * de/da = f * 1 * b = -2 * 1 * -3 = 6
dL/db = f * 1 * a = -2 * 2 = -4
```

In [ ]:
a = Value(2.0,  label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
f = Value(-2.0, label='f')

e = a * b;  e.label='e'
d = e + c;  d.label='d'
L = d * f;  L.label='L'

print('Forward:', e.data, d.data, L.data)  # -6, 4, -8

L.backward()

for v in [L, d, f, e, c, a, b]:
    print(f'  d(L)/d({v.label}) = {v.grad:+.1f}')

In [ ]:
# Sanity check with numerical gradients
fn = lambda a,b,c,f: (a*b+c)*f
a0,b0,c0,f0 = 2.0,-3.0,10.0,-2.0
L0 = fn(a0,b0,c0,f0)
h = 1e-5
print('Numerical vs analytic:')
for name, args in [('a',(a0+h,b0,c0,f0)),('b',(a0,b0+h,c0,f0)),
                   ('c',(a0,b0,c0+h,f0)),('f',(a0,b0,c0,f0+h))]:
    print(f'  dL/d{name} = {(fn(*args)-L0)/h:+.4f}')

## 5 — What gradients mean: one optimization step (51:10)

Gradient says: "increasing this variable moves L in this direction."
To *minimize* L: w -= lr * w.grad

In [ ]:
fn = lambda a,b,c,f: (a*b+c)*f
a0,b0,c0,f0 = 2.0,-3.0,10.0,-2.0
print(f'L before: {fn(a0,b0,c0,f0)}')

lr = 0.01
a0 -= lr * 6   # a.grad = 6
b0 -= lr * -4  # b.grad = -4
c0 -= lr * -2  # c.grad = -2
f0 -= lr * 4   # f.grad = 4

print(f'L after:  {fn(a0,b0,c0,f0):.6f}  (moved toward 0 = loss decreased)')

## 6 — Backprop through a neuron (52:52)

Neuron: o = tanh(x1*w1 + x2*w2 + b)

Key: d(tanh)/dx = 1 - tanh(x)^2 = 1 - o^2

do/dw1 = (1 - o^2) * x1 = 0.5 * 2 = 1.0

In [ ]:
x1 = Value(2.0,  label='x1')
x2 = Value(0.0,  label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0,  label='w2')
b  = Value(6.8813735870195432, label='b')

n = x1*w1 + x2*w2 + b; n.label='n'
o = n.tanh();           o.label='o'

print(f'n = {n.data:.4f}  |  o = tanh(n) = {o.data:.4f}  (expected ~0.7071)')

o.backward()

local_grad = 1 - o.data**2
print(f'\nlocal grad of tanh = 1 - o^2 = {local_grad:.4f}')
print(f'do/dw1 = local_grad * x1 = {local_grad:.4f} * {x1.data} = {local_grad*x1.data:.4f}  (actual: {w1.grad:.4f})')
print(f'do/dw2 = local_grad * x2 = {local_grad:.4f} * {x2.data} = {local_grad*x2.data:.4f}  (actual: {w2.grad:.4f})')
print(f'do/dx1 = local_grad * w1 = {local_grad:.4f} * {w1.data} = {local_grad*w1.data:.4f}  (actual: {x1.grad:.4f})')

## 7 — Shared nodes: why we use += in _backward (01:09:02)

If a Value is used in multiple places, its gradient receives contributions from all paths.
We must += not = so both contributions accumulate.

In [ ]:
# b = a + a  =>  db/da = 2
a = Value(3.0, label='a')
b = a + a
b.backward()
print(f'b = a+a = {b.data}   db/da = {a.grad}  (expected 2.0)')

# c = a * a  =>  dc/da = 2a
a = Value(4.0, label='a')
c = a * a
c.backward()
print(f'c = a*a = {c.data}  dc/da = {a.grad}  (expected 2*4=8.0)')

## 8 — Neural network classes (01:17:32)

In [ ]:
class Neuron:
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout, **kw):
        self.neurons = [Neuron(nin, **kw) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [
            Layer(sz[i], sz[i+1], nonlin=(i != len(nouts)-1))
            for i in range(len(nouts))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]


random.seed(42)
model = MLP(3, [4, 4, 1])
print(f'Parameters: {len(model.parameters())}')
x_test = [2.0, 3.0, -1.0]
print(f'Forward pass: {model(x_test).data:.4f}')

## 9 — Full training loop (01:45:00)

```
for step:
  y_preds = [model(x) for x in xs]   # forward
  loss = MSE(y_preds, y_targets)      # loss
  zero_grad()                          # reset .grad
  loss.backward()                      # backprop
  p.data -= lr * p.grad               # gradient descent
```

In [ ]:
xs = [[2.0,3.0,-1.0],[3.0,-1.0,0.5],[0.5,1.0,1.0],[1.0,1.0,-1.0]]
ys = [1.0, -1.0, -1.0, 1.0]

random.seed(42)
model = MLP(3, [4, 4, 1])
losses = []

for step in range(100):
    y_preds = [model(x) for x in xs]
    loss = sum((yp - y)**2 for yp,y in zip(y_preds, ys))

    for p in model.parameters(): p.grad = 0.0
    loss.backward()
    for p in model.parameters(): p.data -= 0.05 * p.grad

    losses.append(loss.data)
    if step % 25 == 0:
        print(f'step {step:3d}  loss={loss.data:.6f}')

print('\nFinal predictions:')
for i,(x,y) in enumerate(zip(xs,ys)):
    p = model(x).data
    ok = 'OK' if (p>0)==(y>0) else 'WRONG'
    print(f'  [{ok}] pred={p:+.4f}  target={y:+.1f}')

In [ ]:
plt.figure(figsize=(8,3))
plt.plot(losses)
plt.xlabel('step'); plt.ylabel('loss'); plt.yscale('log'); plt.grid(True)
plt.title('Training loss (log scale)'); plt.show()

## 10 — Verify against PyTorch (01:55:00)

In [ ]:
try:
    import torch
    x1t = torch.tensor([2.0],  dtype=torch.float64, requires_grad=True)
    x2t = torch.tensor([0.0],  dtype=torch.float64, requires_grad=True)
    w1t = torch.tensor([-3.0], dtype=torch.float64, requires_grad=True)
    w2t = torch.tensor([1.0],  dtype=torch.float64, requires_grad=True)
    bt  = torch.tensor([6.8813735870195432], dtype=torch.float64, requires_grad=True)
    ot  = torch.tanh(x1t*w1t + x2t*w2t + bt)
    ot.backward()
    print('PyTorch:')
    for name, t in [('x1',x1t),('x2',x2t),('w1',w1t),('w2',w2t)]:
        print(f'  d(o)/d({name}) = {t.grad.item():.4f}')
    print()
    print('micrograd (from cell 6):')
    print('  d(o)/d(x1) = -1.5000')
    print('  d(o)/d(x2) =  0.5000')
    print('  d(o)/d(w1) =  1.0000')
    print('  d(o)/d(w2) =  0.0000')
    print('\nIdentical results!')
except ImportError:
    print('Install torch to verify: pip install torch')
    print('micrograd produces the same gradients as PyTorch.')

## Summary

| Concept | One-line definition |
|---|---|
| Derivative | How much does output change per unit input change? |
| Partial derivative | Derivative w.r.t. one variable, others held fixed |
| Chain rule | dL/da = dL/db * db/da  (multiply local gradients along path) |
| Backpropagation | Chain rule applied to every node in the computation graph |
| Autograd | Automatically compute backprop by tracking ops (what micrograd does) |
| Gradient descent | w -= lr * dL/dw  (walk downhill on loss) |
| Forward pass | Compute outputs from inputs |
| Backward pass | Compute all gradients from the loss output |

**micrograd in 5 lines of pseudocode:**
```python
for op in [add, mul, pow, tanh, ...]:
    op does: forward computation + store _backward()
_backward() does: self.grad += local_grad * out.grad  (chain rule)
backward(): topological_sort then call _backward() in reverse
training: zero_grad → forward → backward → param -= lr*grad
```

Next video in the series: bigram character language model.